# Train Receipt Models — Target Accuracy > 0.85
**Cải tiến so với version cũ:**
- Dataset lớn hơn: 1540 merchant + 1320 lineitem (gấp ~14 lần)
- Char TF-IDF + Word TF-IDF kết hợp (FeatureUnion)
- Oversampling lớp ít mẫu
- GridSearchCV tối ưu C + class_weight
- Đánh giá chi tiết confusion matrix
- Export model + test trực tiếp trong notebook

In [ ]:
!pip -q install scikit-learn pandas matplotlib seaborn joblib

## Bước 1 — Upload dataset

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

## Bước 2 — Preprocessing + load data

In [ ]:
import random, re, unicodedata
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

# ── Preprocessing ─────────────────────────────────────────
def remove_accents(text):
    text = text.replace('đ','d').replace('Đ','D')
    nfkd = unicodedata.normalize('NFKD', text)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))

def preprocess(text):
    if not isinstance(text, str): return ''
    text = remove_accents(text)
    text = unicodedata.normalize('NFC', text).lower().strip()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def load_dataset(path):
    df = pd.read_csv(path)
    # Hỗ trợ cả 2 tên cột: 'category' và 'label'
    if 'label' in df.columns and 'category' not in df.columns:
        df = df.rename(columns={'label': 'category'})
    df = df[['text','category']].copy()
    df['text'] = df['text'].astype(str).map(preprocess)
    df['category'] = df['category'].astype(str).str.strip()
    df = df[df['text'].str.len() >= 2].reset_index(drop=True)
    return df

# ── Load ──────────────────────────────────────────────────
merchant_df = load_dataset('merchant_train.csv')
lineitem_df  = load_dataset('lineitem_train.csv')

print(f'Merchant: {len(merchant_df)} rows | {merchant_df.category.nunique()} classes')
print(f'Lineitem: {len(lineitem_df)} rows | {lineitem_df.category.nunique()} classes')

# Phân bổ
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, df, title in [
    (axes[0], merchant_df, 'Merchant dataset'),
    (axes[1], lineitem_df,  'Lineitem dataset'),
]:
    counts = df['category'].value_counts()
    ax.bar(counts.index, counts.values, color='steelblue')
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', rotation=35)
    for i, v in enumerate(counts.values):
        ax.text(i, v+1, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## Bước 3 — Build pipeline (FeatureUnion char + word TF-IDF)

In [ ]:
def oversample(x, y, min_per_class=60, seed=42):
    """Oversample lớp ít mẫu đến min_per_class"""
    rng = random.Random(seed)
    by_class = {}
    for xi, yi in zip(x, y):
        by_class.setdefault(yi, []).append(xi)
    nx, ny = list(x), list(y)
    for cat, texts in by_class.items():
        needed = max(0, min_per_class - len(texts))
        for _ in range(needed):
            nx.append(rng.choice(texts))
            ny.append(cat)
    return nx, ny

def build_lr_pipeline():
    """Logistic Regression với FeatureUnion char + word"""
    return Pipeline([
        ('features', FeatureUnion([
            ('char', TfidfVectorizer(
                analyzer='char_wb', ngram_range=(2,5),
                min_df=1, sublinear_tf=True, max_features=120_000
            )),
            ('word', TfidfVectorizer(
                analyzer='word', ngram_range=(1,3),
                min_df=1, sublinear_tf=True, max_features=60_000
            )),
        ])),
        ('clf', LogisticRegression(
            max_iter=1000, class_weight='balanced', solver='lbfgs'
        ))
    ])

def build_svm_pipeline():
    """Linear SVM — thường mạnh hơn LR với text ngắn"""
    return Pipeline([
        ('features', FeatureUnion([
            ('char', TfidfVectorizer(
                analyzer='char_wb', ngram_range=(2,5),
                min_df=1, sublinear_tf=True, max_features=120_000
            )),
            ('word', TfidfVectorizer(
                analyzer='word', ngram_range=(1,3),
                min_df=1, sublinear_tf=True, max_features=60_000
            )),
        ])),
        # CalibratedClassifierCV để có predict_proba (confidence score)
        ('clf', CalibratedClassifierCV(
            LinearSVC(max_iter=3000, class_weight='balanced'), cv=3
        ))
    ])

print('✅ Pipeline sẵn sàng!')

## Bước 4 — Train + GridSearchCV + Đánh giá

In [ ]:
def train_and_eval(df, task_name, use_tuning=True):
    print(f'\n{"="*55}\n🚀 {task_name}\n{"="*55}')

    x = df['text'].tolist()
    y = df['category'].tolist()

    # Split
    x_tr, x_te, y_tr, y_te = train_test_split(
        x, y, test_size=0.2, random_state=42, stratify=y
    )

    # Oversample train set
    x_tr, y_tr = oversample(x_tr, y_tr, min_per_class=60)
    print(f'Train size after oversample: {len(x_tr)}')

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # ── So sánh LR vs SVM ────────────────────────────────
    results = {}
    print('\nSo sánh model (CV macro-F1):')
    for name, pipe in [('LogisticRegression', build_lr_pipeline()),
                        ('LinearSVM',         build_svm_pipeline())]:
        from sklearn.model_selection import cross_val_score
        scores = cross_val_score(pipe, x, y, cv=cv,
                                 scoring='f1_macro', n_jobs=-1)
        results[name] = {'pipe': pipe, 'cv': scores.mean(), 'std': scores.std()}
        print(f'  {name:<25} {scores.mean():.4f} ± {scores.std():.4f}')

    best_name = max(results, key=lambda k: results[k]['cv'])
    best_pipe = results[best_name]['pipe']
    print(f'\n🏆 Best: {best_name}')

    # ── GridSearchCV trên model tốt nhất ─────────────────
    if use_tuning:
        print('\n⚙️  GridSearchCV...')
        if 'LogisticRegression' in best_name:
            param_grid = {
                'clf__C'           : [1.0, 3.0, 6.0, 10.0, 20.0],
                'clf__class_weight': ['balanced', None],
            }
        else:  # SVM
            param_grid = {
                'features__char__ngram_range': [(2,4),(2,5)],
                'features__word__ngram_range': [(1,2),(1,3)],
            }
        gs = GridSearchCV(
            best_pipe, param_grid,
            scoring='f1_macro', cv=cv, n_jobs=-1, verbose=0
        )
        gs.fit(x_tr, y_tr)
        final = gs.best_estimator_
        print(f'  Best params: {gs.best_params_}')
        print(f'  Best CV F1 : {gs.best_score_:.4f}')
    else:
        best_pipe.fit(x_tr, y_tr)
        final = best_pipe

    # ── Đánh giá trên test set ────────────────────────────
    y_pred = final.predict(x_te)
    acc     = accuracy_score(y_te, y_pred)
    macro   = f1_score(y_te, y_pred, average='macro')
    weighted= f1_score(y_te, y_pred, average='weighted')

    print(f'\n📊 Test Accuracy  : {acc:.4f}')
    print(f'   Macro F1       : {macro:.4f}')
    print(f'   Weighted F1    : {weighted:.4f}')
    print('\n' + classification_report(y_te, y_pred, digits=4, zero_division=0))

    # ── Confusion matrix ──────────────────────────────────
    labels = sorted(set(y))
    cm = confusion_matrix(y_te, y_pred, labels=labels)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    for ax, data, fmt, title in [
        (axes[0], cm,      'd',    f'{task_name} — Count'),
        (axes[1], cm_norm, '.1%',  f'{task_name} — Normalized'),
    ]:
        sns.heatmap(data, annot=True, fmt=fmt,
                    xticklabels=labels, yticklabels=labels,
                    cmap='Blues' if ax==axes[0] else 'RdYlGn',
                    ax=ax, linewidths=0.3)
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=35, ha='right')
    plt.tight_layout()
    plt.show()

    # ── Top cặp nhầm ──────────────────────────────────────
    print('🔴 Top cặp bị nhầm nhiều nhất:')
    confused = []
    for i, tl in enumerate(labels):
        for j, pl in enumerate(labels):
            if i != j and cm[i][j] > 0:
                confused.append((cm[i][j], tl, pl))
    for cnt, tl, pl in sorted(confused, reverse=True)[:5]:
        pct = cnt / cm[labels.index(tl)].sum() * 100
        print(f'  {tl:<15} → {pl:<15} {cnt:>3} lần ({pct:.0f}%)')

    return final

print('✅ Hàm train_and_eval() sẵn sàng!')

## Bước 5 — Train Merchant model

In [ ]:
USE_TUNING = True  # False = nhanh hơn, accuracy thấp hơn chút
merchant_model = train_and_eval(merchant_df, 'Merchant Classifier', USE_TUNING)

## Bước 6 — Train Lineitem model

In [ ]:
lineitem_model = train_and_eval(lineitem_df, 'Lineitem Classifier', USE_TUNING)

## Bước 7 — Test thực tế trước khi lưu

In [ ]:
THRESHOLD = 0.40

def classify(text, model):
    clean = preprocess(text)
    cat   = model.predict([clean])[0]
    proba = model.predict_proba([clean])[0]
    conf  = float(max(proba))
    low   = conf < THRESHOLD
    return {'category': 'Khac' if low else cat,
            'confidence': round(conf, 3), 'low': low}

MERCHANT_TESTS = [
    ('Highlands Coffee Vincom',      'An_uong'),
    ('GrabCar di san bay',           'Di_lai'),
    ('Hoc phi ILA thang 9',          'Hoc_tap'),
    ('Tien dien EVN thang 5',        'Gia_dinh'),
    ('Nha thuoc Long Chau',          'Suc_khoe'),
    ('Tiem nail Ruby Quan 3',        'Lam_dep'),
    ('Thu y kham cho meo',           'Thu_cung'),
    ('CGV ve xem phim',              'Giai_tri'),
    ('Shopee mua giay Nike',         'Mua_sam'),
    ('Tour Da Lat 3 ngay',           'Du_lich'),
    ('J&T Express phi van chuyen',   'Khac'),
    ('Ca phe Xin Chao quan moi',     'An_uong'),   # tên lạ
    ('Xe om di lam sang nay',        'Di_lai'),    # viết tắt
]

LINEITEM_TESTS = [
    ('Mi goi Hao Hao tom chua cay',  'An_uong'),
    ('Bot giat Omo 3kg',             'Gia_dinh'),
    ('Vitamin C 1000mg Nature Made', 'Suc_khoe'),
    ('Kem duong am Neutrogena',      'Lam_dep'),
    ('Thuc an meo Whiskas 1.2kg',    'Thu_cung'),
    ('The game Steam Wallet 500k',   'Giai_tri'),
    ('Ao thun cotton Uniqlo',        'Mua_sam'),
    ('Xang RON 95 10L',              'Di_lai'),
    ('Sach Cambridge IELTS 17',      'Hoc_tap'),
    ('Ba lo du lich 40L chong nuoc', 'Du_lich'),
    ('Phong bi thu trang A4',        'Khac'),
]

def run_test(tests, model, title):
    print(f'\n{title}')
    print(f'{"Input":<40} {"Expected":<15} {"Got":<15} {"Conf":>6}  OK?')
    print('─'*85)
    correct = 0
    for text, expected in tests:
        r = classify(text, model)
        ok = r['category'] == expected
        correct += ok
        icon = '✅' if ok else '❌'
        print(f'{text:<40} {expected:<15} {r["category"]:<15} {r["confidence"]:>5.0%}  {icon}')
    print(f'\nKết quả: {correct}/{len(tests)} = {correct/len(tests):.0%}')

run_test(MERCHANT_TESTS, merchant_model, '=== MERCHANT MODEL ===')
run_test(LINEITEM_TESTS,  lineitem_model,  '=== LINEITEM MODEL ===')

## Bước 8 — Lưu model + download

In [ ]:
joblib.dump(merchant_model, 'merchant_classifier_latest.pkl')
joblib.dump(lineitem_model,  'lineitem_classifier_latest.pkl')
print('✅ Saved: merchant_classifier_latest.pkl')
print('✅ Saved: lineitem_classifier_latest.pkl')

from google.colab import files
files.download('merchant_classifier_latest.pkl')
files.download('lineitem_classifier_latest.pkl')

## Bước 9 — Deploy (copy 2 file .pkl vào server)

Sau khi download xong:
1. Copy 2 file `.pkl` vào thư mục `receipt-api/models/`
2. Push lên GitHub → server tự deploy lại
3. Test endpoint `/classify` và `/receipt`

**Hoặc** nếu dùng Colab làm server luôn:
```python
!pip install pyngrok fastapi uvicorn
from pyngrok import ngrok
public_url = ngrok.connect(8000)
print(f'API URL: {public_url}')
!uvicorn main:app --host 0.0.0.0 --port 8000
```